# Binary Classification for High-Dimensional Single-Cell Data

This notebook demonstrates how to use the binary classification module for high-dimensional single-cell genomics data. The module is designed to handle the challenges of high-dimensional data through dimensionality reduction, feature selection, and cross-validation.

## Use Cases

1. **Cell Type Classification**: Classify cells as tumor vs. normal
2. **Treatment Response**: Identify responsive vs. non-responsive cells
3. **Spatial Classification**: Classify cells based on spatial location (e.g., core vs. periphery)
4. **Gene Expression Signature**: Classify based on specific molecular signatures

## Key Features

- Handles high-dimensional data (thousands of features)
- Automatic dimensionality reduction with PCA
- Feature selection capabilities
- Multiple classifier options (Logistic Regression, Random Forest, SVM)
- Cross-validation for robust performance estimation
- Integration with AnnData objects

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Import our binary classification module
from binary_classification import HighDimBinaryClassifier, classify_anndata, example_usage

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries loaded successfully!")

## Example 1: Basic Usage with Synthetic High-Dimensional Data

Let's start with a synthetic dataset that mimics the characteristics of single-cell RNA-seq data.

In [ ]:
# Generate synthetic high-dimensional data similar to single-cell RNA-seq
# This simulates having ~2000 genes and 1000 cells
X, y = make_classification(
    n_samples=1000,
    n_features=2000,  # Simulate ~2000 genes
    n_informative=100,  # 100 genes are actually informative
    n_redundant=200,    # 200 genes are redundant
    n_clusters_per_class=2,  # Multiple clusters per class (common in biology)
    flip_y=0.01,        # Small amount of label noise
    random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Features (genes): {X.shape[1]}")
print(f"Samples (cells): {X.shape[0]}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Visualize class distribution
plt.figure(figsize=(6, 4))
unique, counts = np.unique(y, return_counts=True)
plt.bar(['Class 0', 'Class 1'], counts)
plt.title('Class Distribution')
plt.ylabel('Number of Samples')
plt.show()

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

### Classifier 1: Logistic Regression with PCA

In [ ]:
# Initialize classifier with PCA for dimensionality reduction
classifier_lr = HighDimBinaryClassifier(
    classifier_type='logistic',
    n_components=50,  # Reduce to 50 principal components
    random_state=42
)

# Fit the classifier
print("Fitting Logistic Regression classifier...")
classifier_lr.fit(X_train, y_train)

# Evaluate performance
results_lr = classifier_lr.evaluate(X_test, y_test)

print(f"\nLogistic Regression Results:")
print(f"Accuracy: {results_lr['accuracy']:.3f}")
print(f"ROC AUC: {results_lr['roc_auc']:.3f}")
print(f"CV Score (mean ± std): {results_lr['cv_mean_score']:.3f} ± {results_lr['cv_std_score']:.3f}")

In [ ]:
# Plot performance
classifier_lr.plot_performance(X_test, y_test)

### Classifier 2: Random Forest with Feature Selection

In [ ]:
# Initialize Random Forest classifier with feature selection
classifier_rf = HighDimBinaryClassifier(
    classifier_type='random_forest',
    n_components=100,      # More components for Random Forest
    feature_selection_k=50, # Select top 50 features after PCA
    random_state=42
)

# Fit the classifier
print("Fitting Random Forest classifier...")
classifier_rf.fit(X_train, y_train)

# Evaluate performance
results_rf = classifier_rf.evaluate(X_test, y_test)

print(f"\nRandom Forest Results:")
print(f"Accuracy: {results_rf['accuracy']:.3f}")
print(f"ROC AUC: {results_rf['roc_auc']:.3f}")
print(f"CV Score (mean ± std): {results_rf['cv_mean_score']:.3f} ± {results_rf['cv_std_score']:.3f}")

In [ ]:
# Plot performance and feature importance
classifier_rf.plot_performance(X_test, y_test)

# Show top features
feature_importance = classifier_rf.get_feature_importance(top_k=10)
print("\nTop 10 Most Important Features:")
print(feature_importance)

### Comparing Multiple Classifiers

In [ ]:
# Compare different classifier types
classifiers = {}
results_comparison = {}

for clf_type in ['logistic', 'random_forest', 'svm']:
    print(f"\nTraining {clf_type} classifier...")
    
    clf = HighDimBinaryClassifier(
        classifier_type=clf_type,
        n_components=50,
        random_state=42
    )
    
    clf.fit(X_train, y_train)
    results = clf.evaluate(X_test, y_test)
    
    classifiers[clf_type] = clf
    results_comparison[clf_type] = results
    
    print(f"ROC AUC: {results['roc_auc']:.3f}")
    print(f"Accuracy: {results['accuracy']:.3f}")

In [ ]:
# Create comparison plot
comparison_df = pd.DataFrame({
    'Classifier': list(results_comparison.keys()),
    'ROC_AUC': [results['roc_auc'] for results in results_comparison.values()],
    'Accuracy': [results['accuracy'] for results in results_comparison.values()],
    'CV_Mean': [results['cv_mean_score'] for results in results_comparison.values()]
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC AUC comparison
axes[0].bar(comparison_df['Classifier'], comparison_df['ROC_AUC'])
axes[0].set_title('ROC AUC Comparison')
axes[0].set_ylabel('ROC AUC')
axes[0].set_ylim(0, 1)

# Accuracy comparison
axes[1].bar(comparison_df['Classifier'], comparison_df['Accuracy'])
axes[1].set_title('Accuracy Comparison')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)

# CV Score comparison
axes[2].bar(comparison_df['Classifier'], comparison_df['CV_Mean'])
axes[2].set_title('Cross-Validation Score')
axes[2].set_ylabel('CV Mean ROC AUC')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\nComparison Summary:")
print(comparison_df.round(3))

## Example 2: Simulating Cell Type Classification

Let's create a more realistic example that simulates classifying tumor cells vs. normal cells.

In [ ]:
# Create synthetic data that mimics single-cell RNA-seq
# Simulate a scenario with tumor vs normal cells
np.random.seed(42)

n_cells = 800
n_genes = 1500

# Create base expression matrix (log-normalized counts)
base_expression = np.random.lognormal(mean=0, sigma=1, size=(n_cells, n_genes))

# Add tumor-specific signature to half the cells
tumor_cells = n_cells // 2
tumor_signature_genes = np.random.choice(n_genes, size=50, replace=False)

# Enhance expression of tumor signature genes in tumor cells
base_expression[:tumor_cells, tumor_signature_genes] *= np.random.uniform(2, 5, size=50)

# Create labels (0 = normal, 1 = tumor)
cell_labels = np.array([1]*tumor_cells + [0]*(n_cells - tumor_cells))

# Add some noise to make it more realistic
noise = np.random.normal(0, 0.1, base_expression.shape)
expression_matrix = base_expression + noise

# Log-transform (common in single-cell analysis)
expression_matrix = np.log1p(expression_matrix)

print(f"Simulated single-cell dataset:")
print(f"Cells: {n_cells}, Genes: {n_genes}")
print(f"Tumor cells: {tumor_cells}, Normal cells: {n_cells - tumor_cells}")
print(f"Tumor signature genes: {len(tumor_signature_genes)}")

In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Expression distribution
axes[0].hist(expression_matrix[cell_labels==0].flatten(), alpha=0.5, label='Normal', bins=50)
axes[0].hist(expression_matrix[cell_labels==1].flatten(), alpha=0.5, label='Tumor', bins=50)
axes[0].set_xlabel('Log Expression')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Expression Distribution')
axes[0].legend()

# Mean expression of tumor signature genes
normal_signature = expression_matrix[cell_labels==0][:, tumor_signature_genes].mean(axis=1)
tumor_signature = expression_matrix[cell_labels==1][:, tumor_signature_genes].mean(axis=1)

axes[1].hist(normal_signature, alpha=0.5, label='Normal', bins=20)
axes[1].hist(tumor_signature, alpha=0.5, label='Tumor', bins=20)
axes[1].set_xlabel('Mean Tumor Signature Expression')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Tumor Signature Genes')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Apply binary classification to the simulated single-cell data
X_train, X_test, y_train, y_test = train_test_split(
    expression_matrix, cell_labels, test_size=0.25, random_state=42, stratify=cell_labels
)

# Use Random Forest classifier (good for gene expression data)
sc_classifier = HighDimBinaryClassifier(
    classifier_type='random_forest',
    n_components=100,  # Keep more components for complex data
    feature_selection_k=200,  # Select top 200 genes
    random_state=42
)

print("Training classifier on simulated single-cell data...")
sc_classifier.fit(X_train, y_train)

# Evaluate
sc_results = sc_classifier.evaluate(X_test, y_test)

print(f"\nSingle-Cell Classification Results:")
print(f"Accuracy: {sc_results['accuracy']:.3f}")
print(f"ROC AUC: {sc_results['roc_auc']:.3f}")
print(f"CV Score: {sc_results['cv_mean_score']:.3f} ± {sc_results['cv_std_score']:.3f}")

print(f"\nClassification Report:")
print(sc_results['classification_report'])

In [ ]:
# Plot performance
sc_classifier.plot_performance(X_test, y_test)

# Show most important genes
gene_names = [f'Gene_{i}' for i in range(n_genes)]
important_genes = sc_classifier.get_feature_importance(feature_names=gene_names, top_k=15)

print("\nTop 15 Most Important Genes:")
print(important_genes)

## Example 3: Working with AnnData Objects

The module includes a convenience function for working directly with AnnData objects, which are standard in single-cell analysis.

In [ ]:
# Try to import anndata, install if not available
try:
    import anndata as ad
    anndata_available = True
except ImportError:
    print("Installing anndata...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'anndata'])
    import anndata as ad
    anndata_available = True

if anndata_available:
    # Create an AnnData object from our simulated data
    adata = ad.AnnData(X=expression_matrix)
    
    # Add cell type labels
    adata.obs['cell_type'] = ['Tumor' if label == 1 else 'Normal' for label in cell_labels]
    
    # Add some metadata
    adata.obs['sample_id'] = [f'Sample_{i//100}' for i in range(n_cells)]
    adata.var['gene_name'] = [f'Gene_{i}' for i in range(n_genes)]
    
    print(f"AnnData object created:")
    print(adata)
    print(f"\nCell type distribution:")
    print(adata.obs['cell_type'].value_counts())
else:
    print("AnnData not available, skipping this example")

In [ ]:
if anndata_available:
    # Use the convenience function for AnnData objects
    classifier, results = classify_anndata(
        adata=adata,
        target_column='cell_type',
        positive_class='Tumor',
        classifier_type='random_forest',
        n_components=50,
        feature_selection_k=100,
        test_size=0.3,
        random_state=42
    )
    
    print("AnnData Classification Results:")
    print(f"Positive class: {results['positive_class']}")
    print(f"Target column: {results['target_column']}")
    print(f"Total samples: {results['n_samples']}")
    print(f"Total features: {results['n_features']}")
    print(f"Class distribution: {results['class_distribution']}")
    print(f"Accuracy: {results['accuracy']:.3f}")
    print(f"ROC AUC: {results['roc_auc']:.3f}")
    print(f"CV Score: {results['cv_mean_score']:.3f} ± {results['cv_std_score']:.3f}")

## Key Takeaways and Best Practices

1. **Dimensionality Reduction**: PCA is essential for high-dimensional single-cell data to reduce noise and computational complexity.

2. **Feature Selection**: Selecting the most informative features can improve performance and interpretability.

3. **Cross-Validation**: Always use cross-validation to get robust performance estimates.

4. **Classifier Choice**: 
   - Logistic Regression: Fast, interpretable, good baseline
   - Random Forest: Handles non-linear relationships, provides feature importance
   - SVM: Good for complex decision boundaries

5. **Data Preprocessing**: Proper scaling and normalization are crucial for good performance.

6. **Evaluation Metrics**: ROC AUC is particularly useful for binary classification, especially with imbalanced classes.

## Real-World Applications

This binary classification framework can be applied to:

- **Cancer Research**: Classify tumor vs. normal cells
- **Drug Discovery**: Identify treatment-responsive cells
- **Development Biology**: Classify differentiated vs. stem cells
- **Spatial Analysis**: Classify cells by tissue region
- **Time Series**: Classify cells by developmental stage

The modular design allows easy adaptation to different datasets and research questions.